# PROTOCOL: 4-Channel Architecture
- 4-Channel
- Visualization
- K-Fold CV
- High-Resolution Patching
- TTA Submission


## Imports

In [ ]:
import os
import csv
import random
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr
from sklearn.utils.class_weight import compute_class_weight
from sklearn.manifold import TSNE

import tifffile as tiff 
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torchvision.transforms.functional as TF
from torchvision.transforms import RandAugment
from torch.cuda.amp import autocast, GradScaler # Mixed Precision for speed

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

## Configuration and Helpers

In [ ]:
class Config:
    SEED = 42
    
    # TRAIN PATHS (Cleaned Data)
    CLEAN_IMG_DIR = '/kaggle/input/dataset/train_data_cleaned/images'
    CLEAN_MASK_DIR = '/kaggle/input/dataset/train_data_cleaned/masks'
    CLEAN_LABELS  = '/kaggle/input/dataset/train_data_cleaned/train_labels.csv'
    
    # TIFF PATHS
    TRAIN_TIFF_DIR = '/kaggle/input/tiff-images/train' 
    TEST_TIFF_DIR = '/kaggle/input/tiff-images/test' 
    
    # TEST PATHS
    TEST_IMG_DIR = '/kaggle/input/dataset/test_data/images'
    TEST_MASK_DIR = '/kaggle/input/dataset/test_data/masks'

    
    NUM_CLASSES = 4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    
    # HYPERPARAMS
    IMG_SIZE_BOX = 224   # Low res for finding the tissue (Regressor)
    IMG_SIZE_CLS = 512   # High res for classifying details (Classifier)
    STRIDE = 256         # Overlap for Tiled Inference (50%)
    
    BATCH_SIZE = 16      # Reduce from 32 to fit Lion/ConvNeXt in RAM
    ACCUM_STEPS = 2      # Adjust to maintain Effective BS ~32       

    K_FOLDS = 5
    
    # NEW OPTIMIZER PARAMS
    LR_CLS = 5e-5        # Lion needs very low LR (1e-4 or 5e-5)
    WD_CLS = 1e-2        # Lion needs high decay
    LR_REG = 5e-4        # Ranger for Regressor
    WD_REG = 1e-4
    
    EPOCHS_BOX = 100     # Regressor converges fast (early stopping used)
    EPOCHS_CLS = 500     # Classifier needs more time (early stopping used)
    # DROPOUT = 0.4        # Dropout

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
set_seed(Config.SEED)
print(f"Operational Device: {Config.DEVICE}")

In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, mode="max", verbose=False, delta=0, path='checkpoint.pth', trace_func=print):
        """
        Args:
            patience (int): How long to wait after last time validation metric improved.
            mode (str): One of {"min", "max"}. 
                        "min" -> Stop when metric stops decreasing (e.g. Loss).
                        "max" -> Stop when metric stops increasing (e.g. F1, Accuracy).
            verbose (bool): If True, prints a message for each validation metric improvement.
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
            path (str): Path for the checkpoint to be saved to.
            trace_func (function): trace print function.
        """
        self.patience = patience
        self.mode = mode
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.delta = delta
        self.path = path
        self.trace_func = trace_func

    def __call__(self, val_metric, model):
        score = val_metric

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_metric, model)
            return

        # Check for improvement based on mode
        improved = False
        if self.mode == "min":
            # Improvement if score is LESS than best - delta
            if score < (self.best_score - self.delta):
                improved = True
        elif self.mode == "max":
            # Improvement if score is GREATER than best + delta
            if score > (self.best_score + self.delta):
                improved = True
        
        if improved:
            self.best_score = score
            self.save_checkpoint(val_metric, model)
            self.counter = 0  # Reset counter
        else:
            self.counter += 1
            if self.verbose:
                self.trace_func(f'   EARLY STOPPING: Counter {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, val_metric, model):
        '''Saves model when validation metric improves.'''
        if self.verbose:
            self.trace_func(f'   ✅ Metric improved to {self.best_score:.4f}. Saving model...')
        torch.save(model.state_dict(), self.path)

In [ ]:
def get_bbox_from_mask(mask_np):
    """Returns (x1, y1, x2, y2) from a binary mask."""
    rows = np.any(mask_np, axis=1)
    cols = np.any(mask_np, axis=0)
    if not np.any(rows) or not np.any(cols): return None
    ymin, ymax = np.where(rows)[0][[0, -1]]
    xmin, xmax = np.where(cols)[0][[0, -1]]
    return xmin, ymin, xmax, ymax

### Modern Optimizers: Lion & Ranger

In [ ]:
# Lion Optimizer
class Lion(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        if not 0.0 <= lr: 
            raise ValueError(f"Invalid learning rate: {lr}")
        if not 0.0 <= betas[0] < 1.0: 
            raise ValueError(f"Invalid beta parameter at index 0: {betas[0]}")
        if not 0.0 <= betas[1] < 1.0: 
            raise ValueError(f"Invalid beta parameter at index 1: {betas[1]}")
        
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad(): loss = closure()
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: 
                    continue
                p.data.mul_(1 - group['lr'] * group['weight_decay'])
                grad = p.grad
                state = self.state[p]
                if len(state) == 0: 
                    state['exp_avg'] = torch.zeros_like(p)
                exp_avg = state['exp_avg']
                beta1, beta2 = group['betas']
                update = exp_avg * beta1 + grad * (1 - beta1)
                p.add_(torch.sign(update), alpha=-group['lr'])
                exp_avg.mul_(beta2).add_(grad, alpha=1 - beta2)
        return loss

# Ranger Optimizer
class Ranger(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, alpha=0.5, k=6, betas=(0.9, 0.999), eps=1e-8, weight_decay=0):
        # FIX: Convert generator to list so it can be consumed twice
        params = list(params)
        
        defaults = dict(lr=lr, alpha=alpha, k=k, betas=betas, eps=eps, weight_decay=weight_decay)
        
        # 1. Initialize RAdam with the list
        self.radam = torch.optim.RAdam(params, lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        
        # 2. Initialize Parent Optimizer with the SAME list
        super().__init__(params, defaults)

        # 3. Lookahead initialization
        self.slow_weights = [[p.clone().detach() for p in group['params']] for group in self.param_groups]
        self.step_counter = 0
    
    def step(self, closure=None):
        loss = self.radam.step(closure)
        self.step_counter += 1
        for group, slow_group in zip(self.param_groups, self.slow_weights):
            if self.step_counter % group['k'] == 0:
                alpha = group['alpha']
                for p, slow_p in zip(group['params'], slow_group):
                    if p.grad is None: 
                        continue
                    slow_p.add_(p - slow_p, alpha=alpha)
                    p.data.copy_(slow_p.data)
        return loss

### Otsu Preprocessing Helper

In [ ]:
# find the "True Tissue" so we don't extract empty white patches.
def get_tissue_mask_otsu(img_rgb):
    """
    Generates a binary mask of tissue using Otsu on Saturation channel.
    """
    # 1. Convert to HSV and take Saturation
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    saturation = hsv[:, :, 1]
    
    # 2. Otsu Thresholding
    _, mask = cv2.threshold(saturation, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # 3. Morphology to clean up noise (optional but recommended)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    
    return mask

## The 4-Channel Classifier - Modified
Modified to implement Random High-Res Patching.

In [ ]:
# 4-Channel Dataset
class ClassifierDataset(Dataset):
    def __init__(self, df, img_dir, mask_dir, tiff_dir, transform=None, mode='train', cache_data=True):
        self.df = df
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.tiff_dir = tiff_dir
        self.transform = transform
        self.mode = mode 
        self.le = LabelEncoder()
        self.cache_data = cache_data
        self.cache = []
        
        # Fit on provided df 
        if 'label' in self.df.columns:
            self.df['encoded_label'] = self.le.fit_transform(self.df['label'])

        # Pre-loading sequence for faster training
        if self.cache_data:
            print(f"   Pre-loading {len(df)} 4-Channel images into RAM...")
            for idx in tqdm(range(len(df)), desc="Caching Classifier"):
                self.cache.append(self.load_item(idx))
            print("   ✅ Caching Complete.")

    def __len__(self):
        return len(self.df)
    

    def load_item(self, idx):
        row = self.df.iloc[idx]
        fname = row['sample_index']
        
        # 1. Load RGB
        img_path = os.path.join(self.img_dir, fname)
        # rgb = Image.open(img_path).convert("RGB")
        
        # Use cv2 for speed, convert to RGB
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # 2. Load TIFF (Fallback logic)
        tiff_name = fname.replace(".png", ".tif")
        tiff_path = os.path.join(self.tiff_dir, tiff_name)
        
        if os.path.exists(tiff_path):
            try:
                tif_data = tiff.imread(tiff_path)
                tif_norm = (tif_data - tif_data.min()) / (tif_data.max() - tif_data.min() + 1e-6)
                # extra_channel = Image.fromarray((tif_norm * 255).astype(np.uint8)).convert("L")
                if t.dtype != np.uint8:
                     t = (t / t.max() * 255).astype(np.uint8)
            except: 
                # extra_channel = Image.new('L', rgb.size, 0)
                t = np.zeros(img.shape[:2], dtype=np.uint8)
        else: 
            # extra_channel = Image.new('L', rgb.size, 0)
            t = np.zeros(img.shape[:2], dtype=np.uint8)

        # Stack (H, W, 4)
        if t.ndim == 2: t = np.expand_dims(t, axis=-1)
        full_img = np.concatenate([img, t], axis=-1)
        
        label = row['encoded_label'] if 'encoded_label' in row else -1
        return full_img, label

    def get_random_crop(self, img, crop_size=512):
        h, w = img.shape[:2]
        
        # Pad if smaller
        if h < crop_size or w < crop_size:
            pad_h = max(0, crop_size - h)
            pad_w = max(0, crop_size - w)
            img = np.pad(img, ((0, pad_h), (0, pad_w), (0, 0)), mode='constant')
            h, w = img.shape[:2]
            
        # Random Crop
        y = random.randint(0, h - crop_size)
        x = random.randint(0, w - crop_size)
        return img[y:y+crop_size, x:x+crop_size, :]
        
    def __getitem__(self, idx):
        # 1. Fetch Base Tensor (Instant)
        if self.cache_data:
            full_img_np, label = self.cache[idx]
        else:
            full_img_np, label = self.load_item(idx)
            
        # 2. Apply Augmentation (Manual for 4-channel - dynamically per epoch)
        if self.mode == 'train':
 
            # 1. Geometry
            patch_t = TF.to_tensor(patch)
            if random.random() > 0.5: patch_t = TF.hflip(patch_t)
            if random.random() > 0.5: patch_t = TF.vflip(patch_t)
            if random.random() > 0.5: 
                angle = random.choice([90, 180, 270])
                patch_t = TF.rotate(patch_t, angle)
                
            # 2. Color (RGB Only)
            if random.random() > 0.2:
                rgb = patch_t[:3, :, :]
                rgb = TF.adjust_brightness(rgb, 1 + (random.random()*0.4 - 0.2))
                rgb = TF.adjust_contrast(rgb, 1 + (random.random()*0.4 - 0.2))
                patch_t[:3, :, :] = torch.clamp(rgb, 0, 1)
                
        else:
            # Validation: Resize ensures we evaluate global structure cheaply
            # (Strictly speaking, Tiling is better, but Resizing is faster for Val loop)
            patch = cv2.resize(full_img_np, (Config.IMG_SIZE_CLS, Config.IMG_SIZE_CLS))
            patch_t = TF.to_tensor(patch)

        # 3. NORMALIZATION (APPLY LAST)
        mean = [0.485, 0.456, 0.406, 0.5]
        std  = [0.229, 0.224, 0.225, 0.5]
        # full_img = TF.normalize(full_img, mean=mean, std=std)
        patch_t = TF.normalize(patch_t, mean=mean, std=std)
        
        return patch_t, label

In [ ]:
# Classifier Model
def get_classifier_model(num_classes):
    print("Instantiating Classifier (ConvNeXt-Tiny 4-Channel)...")
    # ConvNeXt Tiny is roughly equal to ResNet50 but much better features
    model = models.convnext_tiny(weights='IMAGENET1K_V1')

    # 1. Modify First Layer (3 -> 4 channels)
    # ConvNeXt starts with: model.features[0][0] which is Conv2d
    old_conv = model.features[0][0]
    new_conv = nn.Conv2d(4, old_conv.out_channels, kernel_size=old_conv.kernel_size, 
                         stride=old_conv.stride, padding=old_conv.padding, bias=old_conv.bias is not None)
    
    with torch.no_grad():
        new_conv.weight[:, :3] = old_conv.weight
        new_conv.weight[:, 3:4] = torch.mean(old_conv.weight, dim=1, keepdim=True)
        if old_conv.bias is not None:
            new_conv.bias = nn.Parameter(old_conv.bias.clone())

    model.features[0][0] = new_conv

    # 2. Modify Head (Classifier)
    # ConvNeXt head structure: model.classifier[2] is the Linear layer
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    
    return model.to(Config.DEVICE)

### K-Fold CV + Transfer Learning

In [ ]:
def train_classifier_kfold(df):
    print(f"\nTraining The Classifier ({Config.K_FOLDS}-Fold CV + Sandwich Strategy)...")
    
    # 1. Prepare K-Fold
    skf = StratifiedKFold(n_splits=Config.K_FOLDS, shuffle=True, random_state=Config.SEED)
    le = LabelEncoder()
    df['encoded_label'] = le.fit_transform(df['label'])
    
    # 2. Storage
    fold_histories = []
    oof_preds = []   # Out-Of-Fold Predictions
    oof_targets = [] # Out-Of-Fold True Labels
    
    # 3. LOOP OVER FOLDS
    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['encoded_label'])):
        print(f"\nSTARTING FOLD {fold+1}/{Config.K_FOLDS}")
        
        # --- DATA SETUP ---
        train_df = df.iloc[train_idx].copy()
        val_df = df.iloc[val_idx].copy()
        
        # Datasets (High-Res Patching)
        # Note: cache_data=False saves RAM if tight on memory with 5 folds
        train_ds = ClassifierDataset(train_df, Config.CLEAN_IMG_DIR, Config.CLEAN_MASK_DIR, Config.TRAIN_TIFF_DIR, mode='train', cache_data=False)
        val_ds   = ClassifierDataset(val_df,   Config.CLEAN_IMG_DIR, Config.CLEAN_MASK_DIR, Config.TRAIN_TIFF_DIR, mode='val', cache_data=False)
        
        train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
        val_loader   = DataLoader(val_ds,   batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
        
        # --- MODEL SETUP ---
        model = get_classifier_model(Config.NUM_CLASSES)
        
        # Calculate Class Weights for THIS fold
        all_labels = train_df['encoded_label'].values
        class_weights = compute_class_weight('balanced', classes=np.unique(all_labels), y=all_labels)
        class_weights = torch.tensor(class_weights, dtype=torch.float).to(Config.DEVICE)
        
        # Loss (Weighted + Smoothing)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
        scaler = torch.cuda.amp.GradScaler()

        # =========================================================
        # STAGE 1: TRANSFER LEARNING - WARMUP (Freeze Middle)
        # =========================================================
        print(f"   Fold {fold+1} Stage 1: Warming up Heads & Adapters...")
        
        # 1. Freeze EVERYTHING
        for param in model.parameters(): param.requires_grad = False
            
        # 2. Unfreeze First Layer (Adapter for 4-Channels)
        # ConvNeXt structure: model.features[0][0]
        for param in model.features[0][0].parameters(): param.requires_grad = True
            
        # 3. Unfreeze Last Layer (Classifier Head)
        # ConvNeXt structure: model.classifier[2]
        for param in model.classifier[2].parameters(): param.requires_grad = True
            
        # Verify trainable params
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        # print(f"   Trainable Params: {trainable:,}")

        # Warmup Optimizer (Standard AdamW is safer for initial stabilization)
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-2)
        
        # Run 15 Epochs Warmup
        model.train()
        for epoch in range(15):
            for i, (imgs, lbls) in enumerate(train_loader):
                imgs, lbls = imgs.to(Config.DEVICE), lbls.to(Config.DEVICE)
                optimizer.zero_grad()
                
                with torch.cuda.amp.autocast():
                    out = model(imgs)
                    loss = criterion(out, lbls)
                    loss = loss / Config.ACCUM_STEPS # Normalize loss
                
                scaler.scale(loss).backward()
                
                if (i+1) % Config.ACCUM_STEPS == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    
        print("   ✅ Warmup Complete.")

        # =========================================================
        # STAGE 2: FULL UNFREEZE
        # =========================================================
        print(f"   Fold {fold+1} Stage 2: Full Training...")
        
        # Unfreeze ALL
        for param in model.parameters(): param.requires_grad = True
        
        # Switch to Lion for the long run
        optimizer = Lion(model.parameters(), lr=Config.LR_CLS, weight_decay=Config.WD_CLS)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS_CLS, eta_min=1e-6)
        
        # Early Stopping per fold: monitors F1 -- to be maximized
        model_path = f"classifier_fold{fold}.pth"
        early_stopping = EarlyStopping(patience=30, mode="max", verbose=False, path=model_path)
        
       # History Dictionary
        history = {
            'train_loss':[], 
            'train_acc':[], 
            'val_loss':[], 
            'val_acc':[], 
            'val_f1':[]
        }
        
        for epoch in range(Config.EPOCHS_CLS):
            # --- TRAIN ---
            model.train()
            t_loss_sum, t_total, t_corr = 0, 0, 0
            
            optimizer.zero_grad() # Reset before accumulation starts
            
            for i, (imgs, lbls) in enumerate(tqdm(train_loader, desc=f"F{fold+1} Ep{epoch+1}", leave=False)):
                imgs, lbls = imgs.to(Config.DEVICE), lbls.to(Config.DEVICE)
                
                with torch.cuda.amp.autocast():
                    out = model(imgs)
                    loss = criterion(out, lbls)
                    loss = loss / Config.ACCUM_STEPS  # Normalize loss
                
                scaler.scale(loss).backward()

                # Step only after accumulation
                if (i+1) % Config.ACCUM_STEPS == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()

                # Stats (undo normalization for logging)
                t_loss_sum += loss.item() * Config.ACCUM_STEPS * imgs.size(0)
                _, preds = torch.max(out, 1)  # Define preds for accuracy calculation
                t_corr += (preds == lbls).sum().item()
                t_total += lbls.size(0)
            
            scheduler.step()
            
            # --- VALIDATE ---
            model.eval()
            v_loss_sum, v_total, v_corr = 0, 0, 0
            all_preds, all_labels_val = [], []
            
            with torch.no_grad():
                for imgs, lbls in val_loader:
                    imgs, lbls = imgs.to(Config.DEVICE), lbls.to(Config.DEVICE)
                    out = model(imgs)
                    loss = criterion(out, lbls)
                    
                    v_loss_sum += loss.item() * imgs.size(0)
                    _, preds = torch.max(out, 1)
                    v_corr += (preds == lbls).sum().item()
                    v_total += lbls.size(0)
                    
                    all_preds.extend(preds.cpu().numpy())
                    all_labels_val.extend(lbls.cpu().numpy())
            
            # Calculate metrics
            t_loss = t_loss_sum/t_total
            t_acc = t_corr / t_total
            v_loss = v_loss_sum/v_total
            v_acc = v_corr / v_total
            v_f1 = f1_score(all_labels_val, all_preds, average='macro')
            
            # History
            history['train_loss'].append(t_loss)
            history['train_acc'].append(t_acc)
            history['val_loss'].append(v_loss)
            history['val_acc'].append(v_acc)
            history['val_f1'].append(v_f1)
            
            print(f"   Ep {epoch+1:3d}/{Config.EPOCHS_CLS}: Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.4f} | Val Loss: {v_loss:.4f} | Val Acc: {v_acc:.4f} | Val F1: {v_f1:.4f}")
            
            early_stopping(v_f1, model)
            if early_stopping.early_stop:
                print(f"   Fold {fold+1} Converged.")
                break
        
        # --- END OF FOLD CLEANUP ---
        fold_histories.append(history)
        
        # Generate OOF Data (Reload Best Model for this fold)
        print(f"   📊 Generating OOF Data for Fold {fold+1}...")
        model.load_state_dict(torch.load(model_path))
        model.eval()
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(Config.DEVICE), lbls.to(Config.DEVICE)
                out = model(imgs)
                _, preds = torch.max(out, 1)
                oof_preds.extend(preds.cpu().numpy())
                oof_targets.extend(lbls.cpu().numpy())
                
        # Free memory
        del model, optimizer, scaler, train_loader, val_loader
        torch.cuda.empty_cache()

    # Calculate Overall CV Score
    oof_f1 = f1_score(oof_targets, oof_preds, average='macro')
    print(f"\n✅ CV Complete. Overall OOF F1: {oof_f1:.4f}")
    
    return le, fold_histories, oof_f1, oof_targets, oof_preds

## Smart Inference (Regressor + Classifier + TTA)

Modified to implement Tiled Inference.

In [ ]:
# Helper to perform the sliding window prediction.
def predict_tiled(model, full_img_np, crop_size=512, stride=256, device='cuda'):
    """Slides window, predicts on patches, averages logits."""
    model.eval()
    h, w = full_img_np.shape[:2]
    
    # Pad image to be divisible by stride
    pad_h = (crop_size - h % crop_size) % crop_size
    pad_w = (crop_size - w % crop_size) % crop_size
    if pad_h > 0 or pad_w > 0:
        full_img_np = np.pad(full_img_np, ((0, pad_h), (0, pad_w), (0, 0)))
        h, w = full_img_np.shape[:2]
        
    patches = []
    
    # Extract tiles
    for y in range(0, h - crop_size + 1, stride):
        for x in range(0, w - crop_size + 1, stride):
            patch = full_img_np[y:y+crop_size, x:x+crop_size, :]
            # Transform
            p_t = TF.to_tensor(patch)
            p_t = TF.normalize(p_t, mean=[0.485, 0.456, 0.406, 0.5], std=[0.229, 0.224, 0.225, 0.5])
            patches.append(p_t)
            
    # Batch Predict (Avoid OOM by processing chunks)
    batch_size = 8
    probs_list = []
    
    with torch.no_grad():
        for i in range(0, len(patches), batch_size):
            batch = torch.stack(patches[i:i+batch_size]).to(device)
            logits = model(batch)
            probs_list.append(F.softmax(logits, dim=1))
            
    # Average all tiles
    all_probs = torch.cat(probs_list, dim=0)
    avg_prob = torch.mean(all_probs, dim=0)
    
    return avg_prob

### No Regressor Version

In [ ]:
def smart_inference_ensemble(le, test_img_dir, test_mask_dir, tiff_dir, f1_score=0.0):
    print("\nPHASE 3: Oracle Inference (Mask -> High-Res Crop -> Ensemble Classifier)...")
    
    # We no longer need the Regressor!
    # regressor.eval() 
    
    # 1. LOAD ENSEMBLE
    models = []
    for i in range(Config.K_FOLDS):
        m = get_classifier_model(Config.NUM_CLASSES)
        m.load_state_dict(torch.load(f"classifier_fold{i}.pth"))
        m.eval()
        models.append(m)
    print(f"   Loaded {len(models)} classifier models.")
    
    # Normalization
    mean_4ch = [0.485, 0.456, 0.406, 0.5]
    std_4ch  = [0.229, 0.224, 0.225, 0.5]
    
    test_files = sorted([f for f in os.listdir(test_img_dir) if (f.endswith('.png'))&(f.startswith('img_'))])
    results = []
    
    with torch.no_grad():
        for fname in tqdm(test_files, desc="Inference"):
            img_path = os.path.join(test_img_dir, fname)
            
            # --- 1. FIND THE MASK ---
            # Handle naming convention
            mask_name_v1 = fname.replace("img_", "mask_")
            
            mask_path = None
            if os.path.exists(os.path.join(test_mask_dir, mask_name_v1)):
                mask_path = os.path.join(test_mask_dir, mask_name_v1)
            elif os.path.exists(os.path.join(test_mask_dir, mask_name_v2)):
                mask_path = os.path.join(test_mask_dir, mask_name_v2)
            
            # Load Image
            rgb_orig = Image.open(img_path).convert("RGB")
            w, h = rgb_orig.size
            
            # --- 2. DETERMINE CROP ---
            if mask_path:
                # Trust the Doctogres (Use the Mask)
                mask = np.array(Image.open(mask_path).convert("L"))
                bbox = get_bbox_from_mask(mask)
                if bbox:
                    x1, y1, x2, y2 = bbox
                else:
                    x1, y1, x2, y2 = 0, 0, w, h # Empty mask fallback
            else:
                # Missing mask fallback (Use whole image)
                x1, y1, x2, y2 = 0, 0, w, h
            
            # Expand context (1.25x) - Important for Patching
            # We want to see a bit of the "border" around the tumor
            if mask_path and bbox:
                cx, cy = (x1+x2)//2, (y1+y2)//2
                bw, bh = int((x2-x1)*1.25/2), int((y2-y1)*1.25/2)
                x1, x2 = max(0, cx-bw), min(w, cx+bw)
                y1, y2 = max(0, cy-bh), min(h, cy+bh)

            # --- 3. HIGH-RES CROP ---
            rgb_crop = rgb_orig.crop((x1, y1, x2, y2))
            
            # Load TIFF & Crop
            tiff_name = fname.replace(".png", ".tif")
            tiff_path = os.path.join(tiff_dir, tiff_name)
            if os.path.exists(tiff_path):
                try:
                    t = Image.fromarray(tiff.imread(tiff_path))
                    tif_crop = t.crop((x1, y1, x2, y2))
                except: tif_crop = Image.new('L', rgb_crop.size, 0)
            else: tif_crop = Image.new('L', rgb_crop.size, 0)
            
            # Stack to Numpy (H, W, 4)
            rgb_np = np.array(rgb_crop)
            tif_np = np.array(tif_crop)
            if tif_np.ndim == 2: tif_np = np.expand_dims(tif_np, axis=-1)
            full_img_np = np.concatenate([rgb_np, tif_np], axis=-1)

            # --- 4. ENSEMBLE + TILED PREDICTION ---
            # (This logic remains exactly the same as your previous correct version)
            
            # Create tiles from the high-res crop
            crop_size = Config.IMG_SIZE_CLS
            stride = 256 # 50% Overlap
            
            # Pad
            h_c, w_c = full_img_np.shape[:2]
            pad_h = (crop_size - h_c % crop_size) % crop_size
            pad_w = (crop_size - w_c % crop_size) % crop_size
            if pad_h > 0 or pad_w > 0:
                full_img_np = np.pad(full_img_np, ((0, pad_h), (0, pad_w), (0, 0)))
                h_c, w_c = full_img_np.shape[:2]
                
            patches = []
            for y in range(0, h_c - crop_size + 1, stride):
                for x in range(0, w_c - crop_size + 1, stride):
                    patch = full_img_np[y:y+crop_size, x:x+crop_size, :]
                    p_t = TF.to_tensor(patch)
                    p_t = TF.normalize(p_t, mean=mean_4ch, std=std_4ch)
                    patches.append(p_t)
            
            if len(patches) == 0: 
                p_t = TF.to_tensor(cv2.resize(full_img_np, (512, 512)))
                patches.append(TF.normalize(p_t, mean=mean_4ch, std=std_4ch))

            batch = torch.stack(patches).to(Config.DEVICE)
            
            # Predict with Ensemble
            final_tile_probs = torch.zeros(len(patches), Config.NUM_CLASSES).to(Config.DEVICE)
            
            for model in models:
                with torch.no_grad():
                    model_probs = []
                    for i in range(0, len(patches), 8):
                        mini_batch = batch[i:i+8]
                        logits = model(mini_batch)
                        model_probs.append(F.softmax(logits, dim=1))
                    
                    final_tile_probs += torch.cat(model_probs, dim=0)
            
            final_tile_probs /= len(models)
            avg_img_prob = torch.mean(final_tile_probs, dim=0)
            
            pred_idx = avg_img_prob.argmax().item()
            results.append([fname, le.inverse_transform([pred_idx])[0]])
            
    # Save
    f1_str = f"{f1_score:.4f}".replace(".", "")
    fname = f"submission_ensemble_f1_{f1_str}.csv"
    pd.DataFrame(results, columns=['sample_index', 'label']).to_csv(fname, index=False)
    print(f"✅ Oracle Submission Saved: {fname}")

## Visualization Methods

In [ ]:
# Plot CLASSIFIER History
def plot_history_cls(history, title_suffix=""):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    ax1.plot(history["train_loss"], label="Train", alpha=0.5)
    ax1.plot(history["val_loss"], label="Val", alpha=0.9)
    ax1.set_title(f"Cross entropy loss{title_suffix}")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(history["train_acc"], label="Train", alpha=0.5)
    ax2.plot(history["val_acc"], label="Val", alpha=0.9)
    ax2.set_title(f"Accuracy{title_suffix}")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # F1 plot for this specific history
    plt.figure(figsize=(7, 5))
    plt.plot(history["val_f1"])
    plt.title(f"Validation F1 (macro){title_suffix}")
    plt.xlabel("Epoch")
    plt.ylabel("F1")
    plt.grid(alpha=0.3)
    plt.show()

# Plot confusion matrix
def plot_oof_confusion_matrix(y_true, y_pred, class_names):
    print("\n📊 Generating Aggregated (OOF) Confusion Matrix...")
    
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={"label": "Count"},
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Aggregated Confusion Matrix ({Config.K_FOLDS}-Fold CV)")
    plt.tight_layout()
    plt.show()
    
    print("\nGlobal Classification Report (OOF):\n")
    print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# Test Visual Demo
def visualize_test_predictions(images, cls_probs, box_preds, class_names, max_images=10):
    """
    Visualizes 
    """
    print("📊 Visualizing Test Predictions...")
    num_img = len(images)
    n = min(max_images, num_img)
    rows = 2
    cols = (n + 1) // 2
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)

    plt.suptitle("Test predictions: classification + localisation",
                 fontsize=18, fontweight="bold")

    for i in range(n):
        ax = axes[i]
        # Copy image to avoid modifying original
        img = images[i].copy() 
        h, w, _ = img.shape

        # Draw Box
        bx = box_preds[i]
        x1 = int(bx[0] * w)
        y1 = int(bx[1] * h)
        x2 = int(bx[2] * w)
        y2 = int(bx[3] * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)

        # Draw Label
        lbl_idx = int(np.argmax(cls_probs[i]))
        conf = float(cls_probs[i][lbl_idx] * 100)
        lbl_name = class_names[lbl_idx]

        ax.imshow(img)
        ax.set_title(f"{lbl_name} {conf:.1f}%", fontsize=12)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

def run_visual_demo(classifier, le, test_img_dir, test_mask_dir, tiff_dir):
    """
    Runs inference on 10 random test images using the Oracle (Mask) strategy.
    Visualizes the Ground Truth Mask box and the Tiled Classifier prediction.
    """
    print("\n📊 Running Visual Demo (Oracle Mode)...")
    classifier.eval()
    
    # 1. Pick 10 Random Images
    all_files = [f for f in os.listdir(test_img_dir) if (f.endswith('.png'))&(f.startswith('img_'))]
    if len(all_files) > 10:
        demo_files = random.sample(all_files, 10)
    else:
        demo_files = all_files
        
    # 2. Prepare Data Containers
    images_np = []
    box_preds = [] # Now storing Mask Bounding Boxes
    cls_probs = []
    
    # Normalization
    mean_4ch = [0.485, 0.456, 0.406, 0.5]
    std_4ch  = [0.229, 0.224, 0.225, 0.5]

    with torch.no_grad():
        for fname in tqdm(demo_files, desc="Demo Inference"):
            img_path = os.path.join(test_img_dir, fname)
            
            # --- A. LOAD IMAGE & MASK ---
            rgb_orig = Image.open(img_path).convert("RGB")
            w, h = rgb_orig.size
            images_np.append(np.array(rgb_orig))
            
            # Find Mask
            mask_name_v1 = fname.replace("img_", "mask_")
            mask_name_v2 = fname.replace(".png", "_mask.png")
            
            mask_path = None
            if os.path.exists(os.path.join(test_mask_dir, mask_name_v1)):
                mask_path = os.path.join(test_mask_dir, mask_name_v1)
            elif os.path.exists(os.path.join(test_mask_dir, mask_name_v2)):
                mask_path = os.path.join(test_mask_dir, mask_name_v2)
            
            # Get Box from Mask
            if mask_path:
                mask = np.array(Image.open(mask_path).convert("L"))
                bbox = get_bbox_from_mask(mask)
                if bbox:
                    x1, y1, x2, y2 = bbox
                else:
                    x1, y1, x2, y2 = 0, 0, w, h
            else:
                x1, y1, x2, y2 = 0, 0, w, h
                
            # Store Normalized Box for Visualization
            box_preds.append([x1/w, y1/h, x2/w, y2/h])
            
            # Expand context (1.25x) - Same as Inference
            if mask_path:
                cx, cy = (x1+x2)//2, (y1+y2)//2
                bw, bh = int((x2-x1)*1.25/2), int((y2-y1)*1.25/2)
                x1, x2 = max(0, cx-bw), min(w, cx+bw)
                y1, y2 = max(0, cy-bh), min(h, cy+bh)

            # --- B. CROP & PREPARE ---
            rgb_crop = rgb_orig.crop((x1, y1, x2, y2))
            
            # TIFF
            tiff_name = fname.replace(".png", ".tif")
            tiff_path = os.path.join(tiff_dir, tiff_name)
            if os.path.exists(tiff_path):
                try:
                    t = Image.fromarray(tiff.imread(tiff_path))
                    tif_crop = t.crop((x1, y1, x2, y2))
                except: tif_crop = Image.new('L', rgb_crop.size, 0)
            else: tif_crop = Image.new('L', rgb_crop.size, 0)
            
            # Stack to Numpy
            rgb_np = np.array(rgb_crop)
            tif_np = np.array(tif_crop)
            if tif_np.ndim == 2: tif_np = np.expand_dims(tif_np, axis=-1)
            full_img_np = np.concatenate([rgb_np, tif_np], axis=-1)

            # --- C. PREDICT (TILED) ---
            prob = predict_tiled(classifier, full_img_np, 
                                 crop_size=Config.IMG_SIZE_CLS, 
                                 stride=256, 
                                 device=Config.DEVICE)
            
            cls_probs.append(prob.cpu().numpy())
            
    # 3. Call the plotting function
    visualize_test_predictions(images_np, cls_probs, box_preds, le.classes_, max_images=10)

In [ ]:
def visualize_feature_space(model, loader, le, device):
    """
    Extracts features from the penultimate layer of ConvNeXt and projects them using t-SNE.
    Replaces UMAP to avoid dependency crashes while providing the same diagnostic value.
    """
    print("\n🎨 Projecting Feature Space with t-SNE...")
    model.eval()
    features_list = []
    labels_list = []
    
    # 1. Extract Embeddings
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc="Extracting Features"):
            imgs = imgs.to(device)
            
            # --- Manual Forward Pass (ConvNeXt Specific) ---
            # Forward pass up to the layer BEFORE the classifier
            x = model.backbone.features(imgs)
            x = model.backbone.avgpool(x)
            x = model.backbone.classifier[0](x) # LayerNorm
            x = model.backbone.classifier[1](x) # Flatten
            
            features_list.append(x.cpu().numpy())
            labels_list.append(lbls.numpy())

    features = np.concatenate(features_list)
    labels = np.concatenate(labels_list)
    
    # 2. Run t-SNE (Robust Alternative to UMAP)
    # limit samples if dataset is huge (>5k) to save time, but 1.4k is fine.
    print("   Computing t-SNE projection...")
    tsne = TSNE(n_components=2, perplexity=30, init='pca', learning_rate='auto', random_state=42)
    embedding = tsne.fit_transform(features)
    
    # 3. Plot
    plt.figure(figsize=(12, 10))
    
    # Create DataFrame for Seaborn
    df_plot = pd.DataFrame(embedding, columns=['DIM1', 'DIM2'])
    df_plot['Label'] = le.inverse_transform(labels)
    
    sns.scatterplot(
        data=df_plot, 
        x='DIM1', 
        y='DIM2', 
        hue='Label', 
        palette='tab10', 
        s=20, 
        alpha=0.7
    )
    
    plt.title(f"ConvNeXt Feature Space (t-SNE)\nTotal Samples: {len(labels)}")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

---

## MAIN EXECUTION

In [ ]:
if os.path.exists(Config.CLEAN_LABELS):
    df = pd.read_csv(Config.CLEAN_LABELS)
else:
    print("❌ Cleaned data not found.")  

In [ ]:
# 1. Train Classifier
le, histories, oof_f1, oof_targets, oof_preds = train_classifier_kfold(df)

In [ ]:
# 2. Inference -- Final Submission
smart_inference_ensemble(le, Config.TEST_IMG_DIR,Config.TEST_TIFF_DIR, f1_score=oof_f1)

In [ ]:
# 3. Visualize Results

# Classifier History and Confusion Matrix
print(f"\nF1 = {best_f1: .4f}")
plot_history_cls(cls_hist, title_suffix=" (Classifier)")
plot_oof_confusion_matrix(oof_targets, oof_preds, le.classes_)

# NEW: UMAP Feature Visualization
visualize_feature_space(get_classifier_model(4).to(Config.DEVICE), val_loader, le, Config.DEVICE)

# Visual Demo (Grid of Images)
run_visual_demo(clss, le, Config.TEST_IMG_DIR, Config.TEST_MASK_DIR, Config.TEST_TIFF_DIR)

---